# SymBro — the full pipeline on Colab (no local install required)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Envee-42/SymBro/blob/main/src/toolkit/symbro_full_pipeline_colab.ipynb)

Runs **every stage** — `symbro query` → `download` → `geometry` → `isolate` →
`rfdiffusion` → `pmpnn` → `predict` — inside one Colab session, on Colab's own GPU.
Unlike [`symbro_rfdiffusion_colab.ipynb`](symbro_rfdiffusion_colab.ipynb) (a narrower
notebook that assumes you already have SymBro running locally and only need to borrow a
GPU for the RFdiffusion step, via a manual bundle upload/download), this notebook
installs SymBro itself, RFdiffusion, ProteinMPNN, and a structure-prediction backend
all in Colab, and then just runs the real `symbro` CLI commands directly — the exact
same commands documented in the main README's Quickstart, cell by cell. There is no
separate "notebook version" of the pipeline logic to keep in sync: every stage below is
the actual `toolkit.pipeline`/`toolkit.cli` code, unmodified.

**Use this notebook when:** you don't have a local GPU or HPC cluster reachable *at
all*, and want to go from "search RCSB" to "validated designs" without installing
anything on your own machine first.

**Use `symbro_rfdiffusion_colab.ipynb` instead when:** you already run SymBro locally
(query/download/geometry/isolate are free, CPU-only steps — no reason to leave your
machine for those) and only occasionally need to borrow a GPU for RFdiffusion itself,
e.g. because your usual local/Singularity/SLURM path is temporarily unavailable.

**Predictors covered here:** Boltz (default — MIT-licensed code *and* weights, no
setup beyond `pip install`) and AlphaFold2/ColabFold (fully permissive licensing, the
classic Colab structure-prediction path). **AlphaFold3 is deliberately not included** —
its model weights must be requested directly from Google under a non-commercial license
and manually uploaded; that one manual, approval-gated step doesn't fit a "run all
cells top to bottom" notebook. See `af3.py`'s own module docstring if you want to wire
it in yourself (the `AF3Job`/`prepare_self_consistency_job()` shape is the same as the
other two backends, and — as of the fix in this project's history — doesn't need a
genetic-database directory either, if you pass `run_data_pipeline=False`).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or any GPU option
offered to you), then run the cells in order. Total setup time (installs) is
~5–8 minutes; RFdiffusion/ProteinMPNN/prediction runtime depends on how many designs
you ask for.

---

## 0. Check a GPU is attached

RFdiffusion hard-requires a real CUDA GPU — it doesn't just run slowly on CPU, it fails
outright. Catching a missing GPU here, before spending several minutes on installs, is
cheaper than finding out several steps in.

In [ ]:
#@title Check GPU is available { display-mode: "form" }
import subprocess

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    print("GPU detected:", out.stdout.strip())
except Exception:
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type, pick a GPU (e.g. T4), "
        "then re-run this notebook from the top."
    )

## 1. Install SymBro

`--no-deps` on purpose: SymBro's own `pyproject.toml` lists `torch>=2.0` as a
dependency (ProteinMPNN needs plain torch — see `pmpnn.py`'s own module docstring), but
Colab already ships a CUDA build of torch that RFdiffusion below needs to match
exactly. Letting pip's resolver anywhere near torch here risks it "helpfully"
reinstalling a generic wheel that breaks RFdiffusion's CUDA build — same reasoning the
RFdiffusion install cell below uses `--no-dependencies` for `dgl`. SymBro's other
dependencies (pandas, gemmi, typer, ...) are lightweight and installed explicitly
instead, skipping torch entirely since Colab's own already satisfies `torch>=2.0`.

In [ ]:
#@title Install SymBro { display-mode: "form" }
!pip install -q --no-deps git+https://github.com/Envee-42/SymBro.git
!pip install -q numpy pandas gemmi matplotlib networkx requests PyYAML rcsb-api typer rich

print("SymBro installed:", subprocess.run(["symbro", "--version"], capture_output=True, text=True).stdout.strip())

## 2. Install RFdiffusion (~3 min)

Same install this project's `symbro_rfdiffusion_colab.ipynb` uses:
[sokrypton/RFdiffusion](https://github.com/sokrypton/RFdiffusion), the fork behind the
well-known ColabDesign RFdiffusion notebook, kept in sync with Colab's own current
torch/CUDA build (rather than RosettaCommons' own general-purpose install steps).
Model weights + noise schedules download in the background via `aria2c` while the
Python install runs in the foreground.

One layout difference from RosettaCommons' own repo, and from what `rfdiffusion.py`
assumes by default: this fork keeps `run_inference.py` at the repo root, not under
`scripts/`. The `installation.yaml` cell below sets `script_path: run_inference.py`
to match — without that override, SymBro would look for
`RFdiffusion/scripts/run_inference.py`, which doesn't exist in this fork.

In [ ]:
#@title Install RFdiffusion { display-mode: "form" }
import os
import time

REQUIRED_WEIGHTS = ["Base_ckpt.pt"]
OPTIONAL_WEIGHTS = ["Complex_base_ckpt.pt"]  # only needed for hotspot-guided (binder-style) jobs
ALL_WEIGHTS = REQUIRED_WEIGHTS + OPTIONAL_WEIGHTS

if not os.path.isdir("params"):
    os.system("apt-get install -y -q aria2 > /dev/null")
    os.makedirs("params", exist_ok=True)
    os.system(
        "(\
        aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
        aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
        aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
        touch params/done.txt) &"
    )

if not os.path.isdir("RFdiffusion"):
    print("Installing RFdiffusion...")
    os.system("git clone -q https://github.com/sokrypton/RFdiffusion.git")
    os.system("pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    # Order matters: RFdiffusion's own setup.py depends on a package literally named
    # "se3-transformer", which doesn't exist on PyPI -- it only resolves once this
    # editable install has registered a local package under that name.
    os.system("cd RFdiffusion/env/SE3Transformer && pip install -q .")
    os.system("pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    os.system("pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx")

if not os.path.isdir("RFdiffusion/models"):
    print("Waiting for model weights + schedules to finish downloading...")
    os.makedirs("RFdiffusion/models", exist_ok=True)
    for _ in range(90):  # ~7.5 min ceiling
        if not any(os.path.isfile(f"{m}.aria2") for m in ALL_WEIGHTS):
            break
        time.sleep(5)

    present = [m for m in ALL_WEIGHTS if os.path.isfile(m)]
    missing_required = [m for m in REQUIRED_WEIGHTS if m not in present]
    missing_optional = [m for m in OPTIONAL_WEIGHTS if m not in present]
    if missing_optional:
        print(f"Note: {missing_optional} did not download -- only a problem if you use "
              f"hotspot-guided (binder-style) contigs.")
    if present:
        os.system(f"mv {' '.join(present)} RFdiffusion/models")
    if os.path.isfile("schedules.zip"):
        os.system("unzip -q schedules.zip && rm schedules.zip")
    if missing_required:
        raise FileNotFoundError(
            f"{missing_required} never finished downloading -- these are required for "
            f"every job. Re-run this cell; if it keeps failing, check "
            f"https://github.com/RosettaCommons/RFdiffusion for a moved URL."
        )

os.environ["DGLBACKEND"] = "pytorch"
print("RFdiffusion ready.")

## 3. Install ProteinMPNN (~30 sec)

[dauparas/ProteinMPNN](https://github.com/dauparas/ProteinMPNN) bundles its own model
weights directly in the repo (`vanilla_model_weights/`) — unlike RFdiffusion, there's no
separate multi-GB download step here. ProteinMPNN itself only needs plain PyTorch (CPU
or GPU — see `pmpnn.py`'s own module docstring), which Colab already has.

In [ ]:
#@title Install ProteinMPNN { display-mode: "form" }
import os

if not os.path.isdir("ProteinMPNN"):
    print("Installing ProteinMPNN...")
    os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")
print("ProteinMPNN ready.")

## 4. Install structure-prediction backends (~1–2 min)

Both Boltz and ColabFold (AlphaFold2) install here regardless of which one you pick
later in step 8 — installing both up front is simpler than conditionally installing
based on a form field you haven't seen yet, and the extra time is small next to the
RFdiffusion install above. Boltz's own model weights download automatically on its
first real run (not here) — expect the first `symbro predict --predictor boltz` call
to be slower than later ones.

In [ ]:
#@title Install Boltz + ColabFold { display-mode: "form" }
!pip install -q boltz
!pip install -q colabfold[alphafold]
print("Boltz + ColabFold ready.")

## 5. Point SymBro at these installs

Everything below runs with `backend: local` — Colab's own VM *is* "local" for every
tool involved, there's no separate machine to reach the way a Singularity image or a
SLURM cluster would need. Absolute paths throughout, so this keeps working regardless
of which directory a later cell happens to `%cd` into.

In [ ]:
#@title Write installation.yaml { display-mode: "form" }
import os

installation_yaml = f'''
rfdiffusion:
  backend: local
  repo_path: {os.path.abspath("RFdiffusion")}
  python_executable: python
  script_path: run_inference.py   # sokrypton/RFdiffusion keeps this at the repo root

proteinmpnn:
  repo_path: {os.path.abspath("ProteinMPNN")}
  python_executable: python

structure_prediction:
  default: boltz

boltz:
  backend: local
  boltz_executable: boltz

alphafold2:
  backend: local
  colabfold_executable: colabfold_batch
'''

with open("installation.yaml", "w") as f:
    f.write(installation_yaml)
print(installation_yaml)

## 6. Search RCSB for candidate structures

Fill in the form, then run this cell and the next. Leave `ENTRY_ID` blank to search by
symmetry/resolution; set it (e.g. `4V6B`) to look up one specific structure directly
instead — see the main README's Quickstart and Advanced querying section for the full
`--criterion`/`--filter` syntax if you need something more specific than this form
covers (you can always drop into a `!symbro query ...` cell of your own).

In [ ]:
#@title Search criteria { display-mode: "form" }
SYMMETRY = "C3"  #@param {type:"string"}
RESOLUTION_MAX = 2.5  #@param {type:"number"}
ENTRY_ID = ""  #@param {type:"string"}

query_args = []
if ENTRY_ID.strip():
    query_args += ["--entry-id", ENTRY_ID.strip()]
if SYMMETRY.strip():
    query_args += ["--symmetry", SYMMETRY.strip()]
if RESOLUTION_MAX:
    query_args += ["--resolution-max", str(RESOLUTION_MAX)]

if not query_args:
    raise ValueError("Set at least SYMMETRY or ENTRY_ID above before running this cell.")

print("symbro query", " ".join(query_args))

In [ ]:
!symbro query {" ".join(query_args)}
!symbro download

## 7. Detect symmetry, pick which one to build from

First a broad pass — cheap, shows every symmetry type actually present. Then set
`SYMMETRY_TYPE` in the form below to one of the types printed (e.g. `C3`); leave it
blank to auto-pick the one with the most total axes found (printed either way, so you
can see what was chosen and why).

In [ ]:
!symbro geometry

In [ ]:
#@title Pick a symmetry type (blank = auto-pick) { display-mode: "form" }
import pandas as pd
from toolkit import pipeline

SYMMETRY_TYPE = ""  #@param {type:"string"}

# Reuses pipeline.detected_symmetry_types() -- the same summary `symbro geometry`
# itself prints when run with no --symmetry-type -- rather than re-deriving it here.
broad = pd.read_pickle(".symbro/geometry.pkl")
summary = pipeline.detected_symmetry_types(broad)
if summary.empty:
    raise RuntimeError("No symmetry detected in any downloaded structure -- try a different search.")
print(summary)

if SYMMETRY_TYPE.strip():
    chosen = SYMMETRY_TYPE.strip()
    print(f"Using manually-set symmetry type: {chosen}")
else:
    by_axes = summary.sort_values("total_axis_count", ascending=False)
    chosen = by_axes.iloc[0]["symmetry_type"]
    print(f"Auto-picked {chosen!r} (highest total_axis_count = {by_axes.iloc[0]['total_axis_count']}). "
          f"Set SYMMETRY_TYPE above and re-run this cell to override.")

SYMMETRY_TYPE = chosen

In [ ]:
!symbro geometry --symmetry-type {SYMMETRY_TYPE}
!symbro isolate

## 8. RFdiffusion

Runs through SymBro's own `local` backend directly (no separate progress-bar
reimplementation here, unlike `symbro_rfdiffusion_colab.ipynb` — this notebook has
SymBro itself installed, so it just uses the same blocking-with-print behavior a local
run on your own machine would show). One job per (assembly, component) row from the
`isolate` step above.

In [ ]:
#@title RFdiffusion settings { display-mode: "form" }
NUM_DESIGNS = 10  #@param {type:"integer"}
DIFFUSER_T = 50  #@param {type:"integer"}

!symbro rfdiffusion --num-designs {NUM_DESIGNS} --diffuser-t {DIFFUSER_T} --backend local

## 9. ProteinMPNN

Generates candidate sequences for the top RFdiffusion design(s) by pLDDT.

In [ ]:
#@title ProteinMPNN settings { display-mode: "form" }
TOP_N_DESIGNS = 1  #@param {type:"integer"}
NUM_SEQ_PER_TARGET = 8  #@param {type:"integer"}

!symbro pmpnn --top-n {TOP_N_DESIGNS} --num-seq-per-target {NUM_SEQ_PER_TARGET}

## 10. Structure prediction (self-consistency screening)

Folds ProteinMPNN's best candidate sequence(s) back and checks the refolded shape
actually matches what RFdiffusion designed (CA-RMSD / pLDDT thresholds below) before
you'd trust the sequence. `boltz` is the default here — MIT-licensed code and weights,
no extra setup. `af2` uses ColabFold's own free hosted MSA search API.

In [ ]:
#@title Predictor settings { display-mode: "form" }
PREDICTOR = "boltz"  #@param ["boltz", "af2"]
MAX_RMSD = 2.0  #@param {type:"number"}
MIN_PLDDT = 70.0  #@param {type:"number"}

!symbro predict --predictor {PREDICTOR} --max-rmsd {MAX_RMSD} --min-plddt {MIN_PLDDT}

## 11. Preview validated designs

Pick a design from the dropdown to render it inline — same preview widget
`symbro_rfdiffusion_colab.ipynb` uses.

In [ ]:
#@title Preview a validated design { display-mode: "form" }
!pip install -q py3Dmol ipywidgets -q

import pandas as pd
import ipywidgets as widgets
import py3Dmol
from IPython.display import display as ipy_display

predict_df = pd.read_pickle(".symbro/predict.pkl")
if predict_df.empty:
    print("No candidate passed screening -- check .symbro/pmpnn.csv, or loosen "
          "MAX_RMSD/MIN_PLDDT in the cell above and rerun `symbro predict`.")
else:
    folded_paths = list(predict_df["folded_path"])

    def show_structure(path):
        view = py3Dmol.view(width=600, height=400)
        with open(path) as f:
            view.addModel(f.read(), "cif" if path.endswith(".cif") else "pdb")
        view.setStyle({"cartoon": {"colorscheme": "chainHetatm"}})
        view.zoomTo()
        return view

    design_dropdown = widgets.Dropdown(options=folded_paths, description="Design:")
    preview_area = widgets.Output()

    def _on_change(change):
        preview_area.clear_output()
        with preview_area:
            show_structure(change["new"]).show()

    design_dropdown.observe(_on_change, names="value")
    ipy_display(design_dropdown, preview_area)
    with preview_area:
        show_structure(folded_paths[0]).show()

    print(predict_df[["assembly_id", "component_id", "candidate_id", "rmsd_to_design", "mean_plddt"]])

## 12. Download your results

Zips `predict.csv` (the validated-design summary) plus every folded structure that
passed screening, and downloads it.

In [ ]:
#@title Download results { display-mode: "form" }
import os
import shutil

from google.colab import files

os.makedirs("results_for_download", exist_ok=True)
shutil.copy(".symbro/predict.csv", "results_for_download/predict.csv")
if not predict_df.empty:
    for p in predict_df["folded_path"]:
        shutil.copy(p, os.path.join("results_for_download", os.path.basename(p)))

shutil.make_archive("symbro_results", "zip", "results_for_download")
print(f"Zipped {len(predict_df)} validated design(s) + predict.csv -> symbro_results.zip")
files.download("symbro_results.zip")

---

## Starting a new search in the same session

Run `!symbro clean` (clears every checkpoint + scratch folder), then jump back up to
step 6 — RFdiffusion/ProteinMPNN/the predictor backends are already installed for the
rest of this Colab session, so only the pipeline cells need rerunning.

## Citation

If this pipeline contributed to a publication, please cite:

- **SymBro** — citation TBD, see `CITATION.cff` in the repository once published.
- **RFdiffusion** — Watson, J.L., Juergens, D., Bennett, N.R. *et al.* De novo design of
  protein structure and function with RFdiffusion. *Nature* 620, 1089–1100 (2023).
- **ProteinMPNN** — Dauparas, J., Anishchenko, I., Bennett, N. *et al.* Robust deep
  learning-based protein sequence design using ProteinMPNN. *Science* 378, 49-56 (2022).
- **Boltz** (if used) — Wohlwend, J. *et al.* Boltz-1: Democratizing Biomolecular
  Interaction Modeling. *bioRxiv* (2024).
- **ColabFold/AlphaFold2** (if used) — Mirdita, M., Schütze, K., Moriwaki, Y. *et al.*
  ColabFold: making protein folding accessible to all. *Nat Methods* 19, 679–682 (2022).

## License

RFdiffusion is BSD-licensed (RosettaCommons); ProteinMPNN is MIT; Boltz is MIT
(code and weights); ColabFold/AlphaFold2 weights are CC BY 4.0 — all commercial-use
permitted. SymBro's own license: Apache-2.0 — see the repository's `LICENSE` file.